# 5. 프로젝트 최종 결론

> **Block 고유 특성으로 사전 Cycle을 예측하고, 7개 Scheduling Result는 Position·시간·Delay 결과로 비교했다.**

이 노트북은 분석 코드보다 프로젝트의 질문이 어떻게 수정됐고 무엇을 확인했는지 종합한다.


## 5.1 처음의 질문

처음에는 `Block Position Cycle`을 작업 완료 후 측정되는 실제 공기에 가깝게 이해했다. 그래서 Block 특성에 DDQN이 선택한 Position과 Initial 상태를 더하면 예측력이 좋아질 것으로 예상하고 Set A/B/C를 구성했다.

- Set A: Block
- Set B: Block + DDQN 선택 Position
- Set C: Block + DDQN 선택 Position + Initial


## 5.2 예상 밖 결과와 두 차례의 가정 수정

Random Forest 비교에서는 Set A가 RMSE 4.489로 가장 좋았고, Set B와 Set C는 각각 5.029와 4.990으로 낮아졌다. 처음에는 이를 단순한 변수 추가 실패로 보았지만, 분석을 이어가면서 두 가지 가정을 다시 검토했다.

### 5.2.1 첫 번째 수정 — Cycle의 의미

처음에는 Cycle을 작업 완료 후 측정된 실제 공기로 이해했다. 그러나 원본 테이블과 관련자료를 다시 확인한 결과 Cycle은 Result가 아니라 Block Table에 미리 존재했다. 이에 Cycle을 **스케줄링 이전에 주어진 기준 처리기간**으로 다시 정의했다.

### 5.2.2 두 번째 수정 — 선택 Position의 역할

DDQN Result를 이용하면 Block과 선택 Position을 정확하게 연결할 수 있다. 하지만 Streamlit에서 7개 Result를 바꾸어 보자 같은 Block도 방법에 따라 서로 다른 Position에 배정되는 모습이 분명하게 나타났다.

> Position 후보 정보는 스케줄링 입력이지만, 특정 Block에 연결된 선택 Position은 Scheduling Method가 만든 결과다.

즉 Join이 가능하다는 사실만으로 선택 Position을 사전 Cycle 예측의 독립변수로 사용할 수 있는 것은 아니었다. 기존 Set C 최종모델과 방법별 Cycle RMSE 비교는 이 시간 순서를 뒤섞고 있었다.

### 5.2.3 시각화의 역할

Streamlit은 완성된 결과를 보여주는 발표화면에 그치지 않았다. Scheduling Method를 바꿀 때 Position과 일정이 달라지는 모습을 직접 확인하면서 기존 분석 가정을 검증하고 논리 오류를 발견한 탐색 도구로도 사용되었다.


## 5.3 수정된 분석 구조

```text
① Block 고유 특성 ──→ Block Position Cycle 예측

② Block / Position / Initial / Cycle
                    ↓
             Scheduling Method
                    ↓
   Position / Planned Start·Finish / Delay
```

- 최종 회귀모델은 Set A로 수정했다.
- Set B/C는 DDQN 결과층 정보를 사후에 붙인 탐색적 진단으로만 남겼다.
- Cycle Stability는 Scheduling 성과지표에서 제외했다.
- 7개 방법은 Position 배정, Makespan과 Delay로 비교했다.


## 5.4 Machine Learning 결과

Block 고유 특성만 사용하는 Set A에서 Random Forest가 가장 좋은 기준 성능을 보였다. 튜닝 전 후보모델의 Test RMSE는 4.489, R²는 0.774였고, GridSearchCV로 선택한 최종 모델은 RMSE 4.558, R² 0.767이었다.

튜닝 후 별도 Test 성능은 소폭 낮아졌다. 하이퍼파라미터 튜닝이 항상 Test 성능 향상을 보장하지 않으므로, 이 차이를 숨기지 않고 함께 보고한다.

`block_start_window`, 크기·면적·중량 등의 Block 일정·물리 특성이 예측에 주로 기여했다.

이 모델의 질문은 명확하다.

> **아직 Position이 배정되지 않은 Block의 사전 기준 처리기간은 어느 정도인가?**

Position·Initial 정보는 최종 모델에 사용하지 않으므로 새로운 Block에도 같은 시점의 정보만으로 예측할 수 있다.


## 5.5 Scheduling Result 비교

- 872개 중 849개 Block은 방법에 따라 선택 Position이 달랐다.
- **EDDQN**은 재계산 Makespan이 가장 짧고 Max Delay가 가장 낮았다.
- **Earliest Start**는 Total Delay와 Delayed Blocks가 가장 낮았다.
- 7개 방법 모두 60개 Position을 사용했지만 Position별 배정 분포와 일정 범위는 달랐다.

따라서 하나의 방법을 절대적으로 최적이라고 부르기보다, 전체 일정 길이·누적 지연·최악 지연·공간 활용 중 어떤 목적을 우선하는지에 따라 선택해야 한다.


## 5.6 게시자 결과와 프로젝트의 의미

게시자 성능표에서 EDDQN은 Makespan·Turnover·Utilization 1위, Delayed Blocks·Average Delay Time 2위로 보고되었다. 본 프로젝트의 Result 직접 집계에서도 EDDQN은 Makespan과 Max Delay에서 가장 낮았고 Total Delay는 2위였다.

프로젝트의 의미는 새로운 임의 점수를 추가한 데 있지 않다. 오히려 **예측 대상과 Scheduling 결과의 생성 순서를 구분하고**, 공개 Result를 Block–Position–시간 관점에서 재구성해 비교 가능한 화면과 산출물로 만든 데 있다.


## 5.7 현재 분석의 한계

- **실적 Cycle 부재**: 생산 후 측정된 실제 처리기간이 없어 사전 Cycle 예측이 현장 실적과 얼마나 일치하는지 검증할 수 없다.
- **Cycle 산정식 미공개**: 비용, 공종, 작업조, 이동·재취급 등이 Cycle 산정에 포함되는지 확인할 수 없다.
- **Initial 해석 제한**: Initial에 기록되지 않은 Position이 실제로 모두 비어 있었는지 확인할 수 없다.
- **Result 의존성**: 7개 결과는 게시자가 제공한 데이터와 실험조건에 기반하므로 다른 조선소·기간·운영조건의 외부 검증이 필요하다.
- **공간 좌표 부재**: Position의 실제 Yard 좌표가 없어 현재 평면도는 개념적 배치다.
- **목적함수 의존성**: Makespan, Delay, Utilization, Cost 등 평가기준에 따라 알고리즘 순위가 달라질 수 있다.



## 5.8 향후 확장

실적 착수·완료, Man-hour, 비용과 자원정보가 확보되면 `사전 Cycle 예측 → Scheduling → 실적 수집 → 잔여기간 재예측 → Rescheduling`의 순환구조를 검증할 수 있다.


## 5.9 프로젝트를 통해 얻은 시사점

- 변수를 많이 넣는 것보다 **예측 시점에 실제로 알 수 있는 변수인지** 확인하는 것이 먼저다.
- 예측모델의 입력과 최적화 알고리즘의 결과를 같은 층위에서 해석하면 안 된다.
- 예상과 다른 성능은 실패가 아니라 데이터 생성 순서와 가정을 재검토하는 단서가 될 수 있다.
- 스케줄링은 목적함수에 따라 최적 방법이 달라지는 다기준 의사결정 문제다.
- 머신러닝은 일정을 자동 확정하기보다 Cycle 추정과 위험 진단을 지원하고, Scheduling Engine은 실행 가능한 일정 후보를 계산하는 역할로 분리하는 편이 타당하다.


## 5.10 최종 메시지

> **좋은 모델은 변수를 많이 사용한 모델이 아니라, 데이터가 생성되는 순서와 실제 의사결정 시점을 지킨 모델이다.**

본 프로젝트는 Block Position Cycle 예측에서 출발했지만, 분석 과정에서 잘못 섞였던 예측과 Scheduling 결과를 분리했다. 최종적으로 Block 고유 특성으로 사전 Cycle을 예측하고, 7개 Scheduling Method는 실제 Position·계획일정·Delay 결과로 비교하는 일관된 구조를 완성했다.


## 5.11 참고 근거

1. [`Docs/어마어마한 논리적 오류 발견 후 서사를 어떻게 할것인가에 대하여....md`](<Docs/어마어마한 논리적 오류 발견 후 서사를 어떻게 할것인가에 대하여....md>) — 두 차례의 가정 수정과 최종 프로젝트 구조
2. [`Docs/어마어마한 논리적 오류 발견.md`](<Docs/어마어마한 논리적 오류 발견.md>) — 데이터 생성 순서와 모델 구조 재검토
3. [`#References/README.md`](<%23References/README.md>) — 입력층·Result 구조와 Delay 활용 범위
4. Jiang et al. (2024), *Knowledge-Based Curved Block Construction Scheduling and Application in Shipbuilding*
5. Hur et al. (2015), *A study on the man-hour prediction system for shipbuilding*
6. Son et al. (2014), *A Heuristic Algorithm for Block Storage Planning in Shipbuilding*
